# VOA-ukr metadata collector

Streams every clip in `robinhad/VOA-ukr` and appends one record to `voa_ukr_meta.json` after each file. The dataset column is only `audio` and `transcription` (train split, about 20,507 clips). There is no speaker field, so `speaker_id` is null.

For each clip the notebook stores container metadata, the original transcription, a **Whisper medium** transcript (`faster-whisper`), and the language Whisper detected for that clip.

Kaggle setup:

- Internet on.
- GPU accelerator on. Whisper medium is too slow on CPU for this set.
- Account secret `HF_TOKEN` with a Hugging Face read token.

The JSON is written to `/kaggle/working/voa_ukr_meta.json` after every clip, so a dead session can still be downloaded from the notebook output. On the next run, the checkpoint cell compares `/kaggle/working/voa_ukr_meta.json` with `/kaggle/input/voa_ukr_meta/voa_ukr_meta.json`. If the output file has fewer records, it is replaced by the input file. Paths that already succeeded are then skipped. A record with `error` is retried and replaced.

One Kaggle GPU session will not finish the full set. Resume until `n_files` reaches the stream length.

In [ ]:
%pip install -q -U datasets huggingface_hub faster-whisper

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

try:
    import torch

    cuda = torch.cuda.is_available()
    name = torch.cuda.get_device_name(0) if cuda else "cpu"
except ImportError:
    cuda = False
    name = "cpu"
print("device", name)

In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

DATASET_ID = "robinhad/VOA-ukr"
SPLIT = "train"
META_NAME = "voa_ukr_meta.json"


def meta_output_path() -> Path:
    working = Path("/kaggle/working")
    if working.is_dir():
        return working / META_NAME
    return Path("dataset") / META_NAME


def empty_meta() -> dict:
    return {
        "dataset": DATASET_ID,
        "split": SPLIT,
        "files": [],
    }


def load_meta(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(payload, list):
        payload = {"files": payload}
    payload.setdefault("dataset", DATASET_ID)
    payload.setdefault("split", SPLIT)
    payload.setdefault("files", [])
    return payload


def find_input_meta() -> Path | None:
    explicit = Path("/kaggle/input/voa_ukr_meta/voa_ukr_meta.json")
    if explicit.is_file():
        return explicit
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    found = [path for path in root.rglob(META_NAME) if path.is_file()]
    if not found:
        return None
    return max(found, key=lambda path: path.stat().st_size)


def processed_count(data: dict) -> int:
    return sum(1 for item in data.get("files", []) if item.get("path"))


def read_meta_file(path: Path | None) -> dict | None:
    if path is None or not path.is_file() or path.stat().st_size == 0:
        return None
    try:
        return load_meta(path)
    except json.JSONDecodeError:
        print(f"unreadable {path}")
        return None


def sync_checkpoint(output_path: Path) -> dict:
    incoming_path = find_input_meta()
    output_data = read_meta_file(output_path)
    input_data = read_meta_file(incoming_path)
    output_n = processed_count(output_data or empty_meta())
    input_n = processed_count(input_data or empty_meta())
    print(f"output processed={output_n} path={output_path}")
    print(f"input processed={input_n} path={incoming_path}")
    if input_data is not None and input_n > output_n:
        save_meta(output_path, input_data)
        print(f"copied {incoming_path} -> {output_path}")
        return input_data
    if output_data is not None:
        print("kept output")
        return output_data
    print("no checkpoint, starting empty")
    return empty_meta()


def load_resume(output_path: Path) -> dict:
    return sync_checkpoint(output_path)


def save_meta(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    data["n_files"] = len(data["files"])
    data["updated"] = datetime.now(timezone.utc).isoformat()
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def labelled(text) -> bool:
    if text is None:
        return False
    return str(text).strip() not in {"", "-"}


def probe_audio(path: Path) -> dict:
    raw = subprocess.check_output(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "format=duration,bit_rate,format_name:stream=sample_rate,channels,codec_name,bit_rate",
            "-of",
            "json",
            str(path),
        ],
        text=True,
    )
    info = json.loads(raw)
    stream = (info.get("streams") or [{}])[0]
    container = info.get("format") or {}

    def as_int(value):
        if value in (None, "", "N/A"):
            return None
        return int(float(value))

    def as_float(value):
        if value in (None, "", "N/A"):
            return None
        return float(value)

    return {
        "sample_rate": as_int(stream.get("sample_rate")),
        "num_channels": as_int(stream.get("channels")),
        "codec": stream.get("codec_name") or container.get("format_name"),
        "bit_rate": as_int(stream.get("bit_rate") or container.get("bit_rate")),
        "duration_seconds": as_float(container.get("duration")),
    }


meta_path = meta_output_path()

In [ ]:
meta = sync_checkpoint(meta_path)
print("checkpoint files", len(meta["files"]), "output", meta_path)

In [ ]:
import numpy as np
from faster_whisper import WhisperModel

device = "cuda" if cuda else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
whisper_model = WhisperModel("medium", device=device, compute_type=compute_type)
print("whisper", device, compute_type)


def load_waveform(path: Path, sampling_rate: int = 16000) -> np.ndarray:
    # ffmpeg, not PyAV. PyAV's to_ndarray uses np.array(..., copy=False), which NumPy 2 rejects.
    raw = subprocess.check_output(
        [
            "ffmpeg",
            "-nostdin",
            "-v",
            "error",
            "-i",
            str(path),
            "-f",
            "f32le",
            "-ac",
            "1",
            "-ar",
            str(sampling_rate),
            "pipe:1",
        ]
    )
    return np.frombuffer(raw, dtype=np.float32).copy()


def transcribe(path: Path):
    audio = load_waveform(path)
    segments, info = whisper_model.transcribe(
        audio,
        beam_size=5,
        vad_filter=True,
        condition_on_previous_text=False,
    )
    text = " ".join(segment.text.strip() for segment in segments).strip()
    probability = info.language_probability
    return text, info.language, None if probability is None else float(probability)

In [ ]:
from datasets import Audio, load_dataset

ds = load_dataset(
    DATASET_ID,
    split=SPLIT,
    streaming=True,
    token=True,
)
ds = ds.cast_column("audio", Audio(decode=False))

In [ ]:
import hashlib

from tqdm.auto import tqdm

meta = load_resume(meta_path)
done = set()
retry_at = {}
for position, item in enumerate(meta["files"]):
    item_path = item.get("path")
    if not item_path:
        continue
    if item.get("error"):
        retry_at[item_path] = position
    else:
        done.add(item_path)
print("skipping", len(done), "retrying", len(retry_at))

for index, example in enumerate(tqdm(ds, total=20507)):
    audio = example["audio"] or {}
    raw = audio.get("bytes")
    file_path = audio.get("path")
    if not file_path:
        file_path = hashlib.sha256(raw or b"").hexdigest()
    if file_path in done:
        continue

    original = example.get("transcription")
    record = {
        "path": file_path,
        "stream_index": index,
        "split": SPLIT,
        "source": DATASET_ID,
        "speaker_id": None,
        "is_labelled": labelled(original),
        "transcription": original,
        "sample_rate": None,
        "duration_seconds": None,
        "num_channels": None,
        "codec": None,
        "bit_rate": None,
        "num_bytes": len(raw) if raw else None,
        "whisper_transcription": None,
        "language": None,
        "language_probability": None,
        "error": None,
    }
    clip_path = None
    try:
        if not raw:
            raise ValueError("audio bytes are missing")
        suffix = Path(file_path).suffix or ".mp3"
        clip_path = Path("/tmp/voa_current").with_suffix(suffix)
        clip_path.write_bytes(raw)
        record.update(probe_audio(clip_path))
        text, language, probability = transcribe(clip_path)
        record.update(
            {
                "whisper_transcription": text,
                "language": language,
                "language_probability": probability,
            }
        )
    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
        print(f"error {file_path}: {record['error']}")
    finally:
        if clip_path is not None:
            clip_path.unlink(missing_ok=True)

    if file_path in retry_at:
        meta["files"][retry_at.pop(file_path)] = record
    else:
        meta["files"].append(record)
    done.add(file_path)
    save_meta(meta_path, meta)
    print(
        f"{len(done)} {file_path} "
        f"sr={record['sample_rate']} dur={record['duration_seconds']} "
        f"language={record['language']}"
    )

print("wrote", meta_path, "files", len(meta["files"]))